# Assignment 5: Extended Long Short-Term Memory (xLSTM)

*Author:* Philipp Seidl

*Copyright statement:* This  material,  no  matter  whether  in  printed  or  electronic  form,  may  be  used  for  personal  and non-commercial educational use only.  Any reproduction of this manuscript, no matter whether as a whole or in parts, no matter whether in printed or in electronic form, requires explicit prior acceptance of the authors.

In this assignment, we will explore the xLSTM architecture, a novel extension of the classic LSTM model. The paper can be found here: https://arxiv.org/abs/2405.04517

#### Background
Recurrent Neural Networks (RNNs), particularly LSTMs, have proven highly effective in various sequence modeling tasks. However, the emergence of Transformers, with their parallel processing capabilities, has shifted the focus away from LSTMs, especially in large-scale language modeling.
The xLSTM architecture aims to bridge this gap by enhancing LSTMs with mechanisms inspired by modern LLMs (e.g. block-structure, residual connections, ...).  Further, it introduces:
- Exponential gating with normalization and stabilization techniques, which improves gradient flow and memory capacity.
- Modifications to the LSTM memory structure, resulting in two variants:
    - sLSTM: Employs a scalar memory with a scalar update rule and a new memory mixing technique through recurrent connections.
    - mLSTM: Features a matrix memory, employs a covariance update rule, and is fully parallelizable, making it suitable for scaling.

By integrating these extensions into residual block backbones, xLSTM blocks are formed, which can then be mixed and stacked to create the xLSTM architecture.

## Exercise 1: Environment Setup

When working with new architectures or specialized frameworks, it's essential to correctly set up the environment to ensure reproducibility. This exercise focuses on setting up the environment for working with the `xlstm` repository.

1. Visit and clone the official repository: [https://github.com/NX-AI/xlstm](https://github.com/NX-AI/xlstm).  
2. Set up the environment  
3. Document your setup:  
   - OS, Python version, Environment setup, CUDA version (if applicable), and GPU details.  
   - Note any challenges you faced and how you resolved them. 
4. Submit your setup as a bash script using the IPython `%%bash` magic. Ensure it is reproducible.

Getting only mLSTM working is sufficient (if you encounter issues with sLSTM cuda kernels)

> **Note**: Depending on your system setup, you may need to adjust the `environment_pt220cu121.yaml` file, such as for the CUDA version. For this assignment, it is recommended to run it on GPUs. If you don't have one, consider using  [Colab](https://colab.research.google.com/notebooks/welcome.ipynb#recent=true) or other online resources.

> **Recommendations**: While the repository suggests using `conda`, we recommend using `mamba` or `micromamba` instead (way faster) (except if you are using colab). Learn more about them here: [https://mamba.readthedocs.io/en/latest/index.html](https://mamba.readthedocs.io/en/latest/index.html).

Questions to prepare: What is ninja, pytorch and cuda compatability and why do we care?

In [6]:
%%bash
########## SOLUTION BEGIN ##########
set -euo pipefail

echo "OS / SYSTEM"
uname -a || true
echo

echo "GPU"
command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi || echo "nvidia-smi not found"
echo

echo "Load conda"
if [ -f "$HOME/miniconda3/etc/profile.d/conda.sh" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
elif [ -f "$HOME/anaconda3/etc/profile.d/conda.sh" ]; then
  source "$HOME/anaconda3/etc/profile.d/conda.sh"
else
  echo "conda.sh not found. Install Miniconda/Anaconda in WSL."
  exit 1
fi
conda --version
echo

REPO_URL="https://github.com/NX-AI/xlstm.git"
WORKDIR="$HOME/projects"
REPO_DIR="$WORKDIR/xlstm"
ENV_NAME="xlstm"

mkdir -p "$WORKDIR"

echo "Clone/update repo (Linux FS)"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" pull --ff-only || true
fi
echo

cd "$REPO_DIR"

ENV_YAML="environment_pt260cu126.yaml"
if [ ! -f "$ENV_YAML" ]; then
  ENV_YAML="environment_pt220cu121.yaml"
fi
echo "Using env file: $ENV_YAML"
echo

echo "Ensure mamba"
if ! command -v mamba >/dev/null 2>&1; then
  conda install -n base -c conda-forge mamba -y
fi
mamba --version
echo

echo "Create/update env (non-interactive)"
if conda env list | awk '{print $1}' | grep -qx "$ENV_NAME"; then
  echo "Env '$ENV_NAME' exists -> updating"
  mamba env update -n "$ENV_NAME" -f "$ENV_YAML" -y
else
  echo "Env '$ENV_NAME' does not exist -> creating"
  mamba env create -n "$ENV_NAME" -f "$ENV_YAML" -y
fi
echo

echo "Install xlstm"
conda run -n "$ENV_NAME" python -m pip install -U pip setuptools wheel
conda run -n "$ENV_NAME" python -m pip install -e . -v
echo

echo "Import test"
conda run -n "$ENV_NAME" python -c "import xlstm; print('xlstm import OK')"
echo

echo "DONE"
########## SOLUTION END ##########

OS / SYSTEM
Linux DESKTOP-74D94MQ 6.6.87.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux

GPU
Thu Jan  8 11:50:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.02              Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 ...    On  |   00000000:01:00.0  On |                  N/A |
|  0%   47C    P0             55W /  250W |     861MiB /   8192MiB |      0% 

warning  libmamba 'repo.anaconda.com', a commercial channel hosted by Anaconda.com, is used.
    
warning  libmamba Please make sure you understand Anaconda Terms of Services.
    
warning  libmamba See: https://legal.anaconda.com/policies/en/



Transaction

  Prefix: /home/luki852/miniconda3/envs/xlstm

  All requested packages already installed


Transaction starting

Transaction finished



warning  libmamba You are using 'pip' as an additional package manager.
    Be aware that packages installed with 'pip' are managed independently from 'conda-forge' channel.



Updating pip packages: --index-url https://download.pytorch.org/whl/cu126, --extra-index-url https://pypi.org/simple, pre-commit, ipykernel, dacite, omegaconf, torchmetrics, tqdm, pytest, pytest-xdist, numpy<2.0, torch, torchvision, torchaudio

Install xlstm

Using pip 25.3 from /home/luki852/miniconda3/envs/xlstm/lib/python3.11/site-packages/pip (python 3.11)
Obtaining file:///home/luki852/projects/xlstm
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for xlstm (pyproject.toml): started
  Building editable for xlstm (pyproje

  Running command installing build dependencies
  Using pip 25.3 from /home/luki852/miniconda3/envs/xlstm/lib/python3.11/site-packages/pip (python 3.11)
    Obtaining dependency information for setuptools>=42 from https://files.pythonhosted.org/packages/a3/dc/17031897dae0efacfea57dfd3a82fdd2a2aeb58e0ff71b77b87e44edc772/setuptools-80.9.0-py3-none-any.whl.metadata
    Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
    Obtaining dependency information for wheel from https://files.pythonhosted.org/packages/0b/2c/87f3254fd8ffd29e4c02732eee68a83a1d3c346ae39bc6822dcbcb697f2b/wheel-0.45.1-py3-none-any.whl.metadata
    Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl (1.2 MB)
  Using cached wheel-0.45.1-py3-none-any.whl (72 kB)
  
    Creating /tmp/pip-build-env-eosex8ig/overlay/bin

    changing mode of /tmp/pip-build-env-eosex8ig/overlay/bin/wheel to 755

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [setupto



Import test
xlstm import OK


DONE


In [7]:
# Verify your installation of xLSTM:
from omegaconf import OmegaConf
from dacite import from_dict
from dacite import Config as DaciteConfig
from xlstm import xLSTMBlockStack, xLSTMBlockStackConfig
import os
import torch

DEVICE = "cuda" if torch.cuda.is_available() else 'cpu'

use_slstm_kernels = False # set to True if you want to check if sLSTM cuda kernels are working

xlstm_cfg = f"""
mlstm_block:
  mlstm:
    conv1d_kernel_size: 4
    qkv_proj_blocksize: 4
    num_heads: 4
slstm_block:
  slstm:
    backend: {'cuda' if use_slstm_kernels else 'vanilla'}
    num_heads: 4
    conv1d_kernel_size: 4
    bias_init: powerlaw_blockdependent
  feedforward:
    proj_factor: 1.3
    act_fn: gelu
context_length: 32
num_blocks: 7
embedding_dim: 64
slstm_at: [] # empty = mLSTM only
"""
cfg = OmegaConf.create(xlstm_cfg)
cfg = from_dict(data_class=xLSTMBlockStackConfig, data=OmegaConf.to_container(cfg), config=DaciteConfig(strict=True))
xlstm_stack = xLSTMBlockStack(cfg)

x = torch.randn(4, 32, 64).to(DEVICE)
xlstm_stack = xlstm_stack.to(DEVICE)
y = xlstm_stack(x)
y.shape == x.shape

True

## Environment Setup – Issues & Pitfalls Encountered

During the setup of the `xlstm` repository, several issues were encountered and resolved. This section documents the main pitfalls and how they were addressed.

### 1. CUDA Toolkit vs. NVIDIA Driver Confusion
- **Issue**: `nvcc: command not found`
- **Cause**: The NVIDIA driver was installed (confirmed via `nvidia-smi`), but the CUDA toolkit (`nvcc`) was not available system-wide.
- **Resolution**: This is expected when using PyTorch CUDA wheels. PyTorch ships its own CUDA runtime, so `nvcc` is not strictly required unless custom CUDA kernels are compiled manually.


### 2. WSL2 Not Properly Enabled
- **Issue**: WslRegisterDistribution failed with error: 0x80370102
- **Cause**: Windows Virtual Machine Platform / virtualization was not fully enabled.
- **Resolution**:
- Enabled **Virtual Machine Platform** in Windows Features
- Verified virtualization support in BIOS
- Confirmed WSL2 kernel was active


### 3. Conda Not Available Inside WSL
- **Issue**: `conda: command not found` inside Ubuntu
- **Cause**: Conda was only installed on Windows, not inside WSL.
- **Resolution**: Installed **Miniconda inside WSL** and initialized conda for bash.


### 4. Conda Terms of Service Blocking Non-Interactive Setup
- **Issue**: CondaToSNonInteractiveError: Terms of Service have not been accepted
- **Cause**: New Conda versions require explicit ToS acceptance.
- **Resolution**:
- bash
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


### 5. mamba Prompting for Environment Overwrite
- **Issue**: `mamba env create` prompted for confirmation (“Overwrite?”) when executed inside a non-interactive `%%bash` cell.
- **Cause**: The conda environment already existed, causing `mamba` to request user input.
- **Resolution**: Modified the setup logic to:
  - Create the environment only if it does not exist
  - Otherwise update the existing environment using `mamba env update`


### 6. Editable Install Failing on Windows-Mounted Filesystem
- **Issue**: during `pip install -e .`
- **Cause**: Editable installs on the Windows-mounted filesystem (`/mnt/c`) caused permission and file-locking issues under WSL.
- **Resolution**: Moved the repository to the Linux filesystem (e.g. `~/projects/xlstm`) before performing the editable install.


### 7. Jupyter Kernel Mismatch
- **Issue**: `ImportError` occurred despite a successful installation of `xlstm`.
- **Cause**: The Jupyter notebook was running with a different Python kernel than the `xlstm` conda environment.
- **Resolution**:
- Installed `ipykernel` in the `xlstm` environment
- Explicitly selected the `xlstm` kernel in JupyterLab


### 8. Local Folder Shadowing Installed Package
- **Issue**: ImportError: cannot import name 'xLSTMBlockStack' from 'xlstm'
- **Cause**: A local folder named `xlstm/` shadowed the installed Python package, causing Python to import the wrong module.
- **Resolution**:
- bash
mv xlstm xlstm_repo


### 9. JupyterLab Browser Not Opening Automatically
- **Issue**: Running jupyter lab did not automatically open a web browser.
- **Cause**: No default browser was configured inside the WSL environment.
- **Resolution**: Manually opened the provided URL (e.g. http://localhost:8888/lab?token=...) in the host system’s browser.



## Exercise 2: Understanding xLSTM Hyperparameters
Explain key hyperparameters that influence the performance and behavior of the xLSTM architecture and explain how they influence total parameter count.
The explanation should include: proj_factor, num_heads, act_fn, context_length, num_blocks, embedding_dim, hidden_size, dropout, slstm_at, qkv_proj_blocksize, conv1d_kernel_size. Also include how the matrix memory size of mLSTM is determined.

The xLSTM architecture extends classical LSTM designs by combining multi-head processing, convolutional projections, and structured memory blocks. The following hyperparameters strongly influence both model behavior and total parameter count.


### `embedding_dim`
- **Description**: Dimensionality of the token embeddings and internal feature representations.
- **Effect on behavior**: Higher values allow richer representations and increased model capacity
- **Effect on parameter count**:
  Most linear layers scale with `embedding_dim²`, making this one of the dominant contributors to total parameters.


### `hidden_size`
- **Description**: Size of the internal hidden state of the LSTM blocks (often equal or proportional to `embedding_dim`).
- **Effect on behavior**: Larger hidden states improve sequence modeling capacity.
- **Effect on parameter count**:
  Gate matrices scale with `hidden_size × hidden_size`, leading to quadratic growth.


### `num_blocks`
- **Description**: Number of stacked xLSTM blocks.
- **Effect on behavior**: Increases model depth and hierarchical feature abstraction.
- **Effect on parameter count**:
  Parameters grow **linearly** with the number of blocks.


### `num_heads`
- **Description**: Number of parallel heads used in mLSTM or sLSTM blocks.
- **Effect on behavior**: Enables multi-subspace processing and improves expressiveness.
- **Effect on parameter count**:
  QKV projections are replicated per head, causing roughly linear scaling with `num_heads`.


### `proj_factor`
- **Description**: Expansion factor used in feedforward or projection layers.
- **Effect on behavior**: Larger values increase intermediate feature dimensionality and nonlinearity.
- **Effect on parameter count**:
  Feedforward layers scale approximately as
  `embedding_dim × (proj_factor × embedding_dim)`.


### `act_fn`
- **Description**: Activation function used in feedforward layers (e.g. GELU, ReLU).
- **Effect on behavior**: Influences smoothness, gradient flow, and training stability.
- **Effect on parameter count**:
  Does not affect parameter count.


### `dropout`
- **Description**: Probability of randomly zeroing activations during training.
- **Effect on behavior**: Regularization, reduces overfitting.
- **Effect on parameter count**:
  No impact on parameter count.


### `context_length`
- **Description**: Maximum sequence length processed by the model.
- **Effect on behavior**: Determines how much temporal context the model can attend to.
- **Effect on parameter count**:
  Does **not** increase trainable parameters, but affects **memory usage and compute cost**.


### `slstm_at`
- **Description**: Specifies at which layers sLSTM blocks replace mLSTM blocks.
- **Effect on behavior**: Enables selective use of structured or CUDA-accelerated sLSTM blocks.
- **Effect on parameter count**:
  Changes parameter distribution depending on the internal structure of sLSTM, but total count remains comparable.


### `qkv_proj_blocksize`
- **Description**: Block size used for chunked QKV projections.
- **Effect on behavior**: Improves memory locality and computational efficiency.
- **Effect on parameter count**:
  Does **not** change the total number of parameters, only how they are grouped.


### `conv1d_kernel_size`
- **Description**: Kernel size of 1D convolutions used in projection or gating mechanisms.
- **Effect on behavior**: Larger kernels capture longer local temporal patterns.
- **Effect on parameter count**:
  Convolutional parameters scale linearly with `kernel_size × embedding_dim`.


## Matrix Memory Size of mLSTM

The defining feature of mLSTM is its **matrix-valued memory** instead of a vector-valued memory.

- For each head, mLSTM maintains a memory matrix of size: hidden_size × hidden_size
- With `num_heads` heads, the total memory per layer becomes: num_heads × hidden_size²
- This matrix memory enables richer associative storage but is the primary reason mLSTM scales more steeply than standard LSTM variants.



In [4]:
########## SOLUTION BEGIN ##########

########## YOUR SOLUTION HERE ##########

## Exercise 3: Train an xLSTM model on the Trump Dataset from the previous exercise
Your task is to train an xLSTM model on the Trump Dataset from the previous exercise. 
- The goal is to achieve an average validation loss $\mathcal{L}_{\text{val}} < 1.35$. 
- You do not need to perform an extensive hyperparameter search, but you should document your runs. Log your runs with used hyperparameters using tools like wandb, neptune, mlflow, ... or a similar setup. Log training/validation loss and learning rate over steps as well as total trainable parameters of the model for each run.
- You can use the training setup from the previous exercises or any setup of your choice using high level training libraries.

In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate xlstm
python -c "import xlstm, torch; print('xlstm ok', xlstm.__version__ if hasattr(xlstm,'__version__') else ''); print('torch', torch.__version__)"
python -c "import wandb" 2>/dev/null || pip install -q wandb
python -c "from torch.utils.tensorboard import SummaryWriter" 2>/dev/null || pip install -q tensorboard

In [8]:
########## SOLUTION BEGIN ##########
import math, time, json, random
from pathlib import Path
from dataclasses import asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -------------------------------------------------------
# Assumes previous cell already executed:
# - OmegaConf, from_dict, DaciteConfig
# - xLSTMBlockStack, xLSTMBlockStackConfig
# - torch, DEVICE
# -------------------------------------------------------

print("DEVICE:", DEVICE)

def seed_all(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# -------------------------
# Dataset: Character-level LM
# -------------------------
class CharLMDataset(Dataset):
    def __init__(self, ids: torch.Tensor, context_length: int):
        self.ids = ids
        self.T = context_length

    def __len__(self):
        return max(0, self.ids.numel() - self.T - 1)

    def __getitem__(self, idx):
        x = self.ids[idx: idx + self.T]
        y = self.ids[idx + 1: idx + 1 + self.T]
        return x, y

def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch:i for i, ch in enumerate(chars)}
    itos = {i:ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict):
    return torch.tensor([stoi[c] for c in text], dtype=torch.long)

# -------------------------
# Model: embedding -> xLSTM -> lm_head
# -------------------------
class xLSTMLM(nn.Module):
    def __init__(self, vocab_size: int, stack_cfg: xLSTMBlockStackConfig, embedding_dim: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.drop = nn.Dropout(dropout)
        self.stack = xLSTMBlockStack(stack_cfg)
        self.head = nn.Linear(embedding_dim, vocab_size, bias=False)

    def forward(self, idx):  # idx: [B,T]
        x = self.embed(idx)       # [B,T,D]
        x = self.drop(x)
        x = self.stack(x)         # [B,T,D]
        logits = self.head(x)     # [B,T,V]
        return logits

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# -------------------------
# LR schedule
# -------------------------
def cosine_warmup_lr(step, total_steps, base_lr, warmup_steps):
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

@torch.no_grad()
def evaluate(model, loader, max_batches=50):
    model.eval()
    losses = []
    for bi, (x, y) in enumerate(loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        losses.append(loss.item())
        if (bi + 1) >= max_batches:
            break
    model.train()
    return float(sum(losses) / max(1, len(losses)))

# -------------------------
# Single run training function
# -------------------------
def train_run(
    run_name: str,
    text_path: str,
    out_dir: str = "./runs_xlstm_trump",
    seed: int = 42,
    # xLSTM / LM hyperparams
    context_length: int = 256,
    embedding_dim: int = 320,
    num_blocks: int = 7,
    num_heads: int = 4,
    proj_factor: float = 1.5,
    act_fn: str = "gelu",
    conv1d_kernel_size: int = 4,
    qkv_proj_blocksize: int = 4,
    slstm_at=None,                 # None -> []
    dropout: float = 0.1,
    # training hyperparams
    batch_size: int = 64,
    lr: float = 2e-3,
    weight_decay: float = 0.1,
    grad_clip: float = 1.0,
    max_steps: int = 5000,
    eval_every: int = 200,
    val_batches: int = 50,
    use_wandb: bool = True,
):
    seed_all(seed)
    out_dir = Path(out_dir)
    log_dir = out_dir / run_name
    log_dir.mkdir(parents=True, exist_ok=True)

    # ---- load text ----
    text = Path(text_path).read_text(encoding="utf-8")
    stoi, itos = build_vocab(text)
    ids = encode(text, stoi)
    vocab_size = len(stoi)

    # ---- split ----
    split = int(0.9 * len(ids))
    train_ids = ids[:split]
    val_ids = ids[split:]

    train_ds = CharLMDataset(train_ids, context_length)
    val_ds   = CharLMDataset(val_ids, context_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)

    # ---- build xLSTM config (mLSTM only) ----
    if slstm_at is None:
        slstm_at = []

    xlstm_cfg = f"""
mlstm_block:
  mlstm:
    conv1d_kernel_size: {conv1d_kernel_size}
    qkv_proj_blocksize: {qkv_proj_blocksize}
    num_heads: {num_heads}
slstm_block:
  slstm:
    backend: vanilla
    num_heads: {num_heads}
    conv1d_kernel_size: {conv1d_kernel_size}
    bias_init: powerlaw_blockdependent
  feedforward:
    proj_factor: {proj_factor}
    act_fn: {act_fn}
context_length: {context_length}
num_blocks: {num_blocks}
embedding_dim: {embedding_dim}
slstm_at: {slstm_at}
"""
    cfg = OmegaConf.create(xlstm_cfg)
    cfg = from_dict(
        data_class=xLSTMBlockStackConfig,
        data=OmegaConf.to_container(cfg),
        config=DaciteConfig(strict=True),
    )

    model = xLSTMLM(vocab_size=vocab_size, stack_cfg=cfg, embedding_dim=embedding_dim, dropout=dropout).to(DEVICE)
    n_params = count_params(model)

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.95))
    warmup_steps = int(0.05 * max_steps)

    # ---- logging backends ----
    metrics_path = log_dir / "metrics.jsonl"
    mf = open(metrics_path, "w", encoding="utf-8")

    wandb_run = None
    if use_wandb:
        try:
            import wandb
            wandb_run = wandb.init(
                project="xlstm-trump",
                name=run_name,
                config=dict(
                    seed=seed,
                    text_path=str(text_path),
                    vocab_size=vocab_size,
                    n_params=n_params,
                    device=DEVICE,
                    context_length=context_length,
                    batch_size=batch_size,
                    embedding_dim=embedding_dim,
                    num_blocks=num_blocks,
                    num_heads=num_heads,
                    proj_factor=proj_factor,
                    act_fn=act_fn,
                    conv1d_kernel_size=conv1d_kernel_size,
                    qkv_proj_blocksize=qkv_proj_blocksize,
                    slstm_at=slstm_at,
                    dropout=dropout,
                    lr=lr,
                    weight_decay=weight_decay,
                    grad_clip=grad_clip,
                    max_steps=max_steps,
                    eval_every=eval_every,
                ),
            )
        except Exception as e:
            print("wandb not available -> logging locally only:", e)
            wandb_run = None

    tb = None
    try:
        from torch.utils.tensorboard import SummaryWriter
        tb = SummaryWriter(log_dir=str(log_dir / "tb"))
    except Exception:
        tb = None

    # ---- training ----
    model.train()
    train_iter = iter(train_loader)
    best_val = float("inf")
    best_ckpt = log_dir / "best.pt"

    t0 = time.time()
    for step in range(1, max_steps + 1):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(DEVICE), y.to(DEVICE)

        cur_lr = cosine_warmup_lr(step-1, max_steps, lr, warmup_steps)
        for pg in optim.param_groups:
            pg["lr"] = cur_lr

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

        optim.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip and grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optim.step()

        # train log every 10 steps
        if step % 10 == 0:
            dt = time.time() - t0
            t0 = time.time()
            msg = dict(step=step, train_loss=float(loss.item()), lr=float(cur_lr), n_params=int(n_params))
            mf.write(json.dumps(msg) + "\n"); mf.flush()
            if wandb_run: wandb_run.log(msg, step=step)
            if tb:
                tb.add_scalar("train/loss", msg["train_loss"], step)
                tb.add_scalar("train/lr", msg["lr"], step)

        # eval
        if step % eval_every == 0:
            val_loss = evaluate(model, val_loader, max_batches=val_batches)
            msg = dict(step=step, val_loss=float(val_loss), lr=float(cur_lr), n_params=int(n_params))
            mf.write(json.dumps(msg) + "\n"); mf.flush()
            if wandb_run: wandb_run.log(msg, step=step)
            if tb: tb.add_scalar("val/loss", val_loss, step)

            print(f"[{run_name}] step={step:5d} train_loss={loss.item():.4f} val_loss={val_loss:.4f} lr={cur_lr:.2e}")

            if val_loss < best_val:
                best_val = val_loss
                torch.save(
                    dict(
                        model_state=model.state_dict(),
                        stoi=stoi, itos=itos,
                        xlstm_cfg=xlstm_cfg,
                        run_name=run_name,
                        n_params=n_params,
                    ),
                    best_ckpt
                )

            # early stop once target is reached
            if best_val < 1.35:
                print(f"[{run_name}] Target reached: best_val={best_val:.4f} < 1.35")
                break

    mf.close()
    if tb: tb.close()
    if wandb_run: wandb_run.finish()

    return best_val, str(best_ckpt)


# -------------------------
# Configure your dataset path here
# -------------------------
TRUMP_TXT_PATH = "trump_train.txt"  # <-- set to your file (absolute or relative)
assert Path(TRUMP_TXT_PATH).exists(), f"File not found: {TRUMP_TXT_PATH}"

# -------------------------
# Documented runs (no big hyperparam search)
# -------------------------
runs = [
    dict(
        run_name="runA_baseline",
        seed=42,
        context_length=256,
        batch_size=64,
        embedding_dim=320,
        num_blocks=7,
        num_heads=4,
        proj_factor=1.5,
        act_fn="gelu",
        conv1d_kernel_size=4,
        qkv_proj_blocksize=4,
        dropout=0.10,
        lr=2e-3,
        weight_decay=0.10,
        grad_clip=1.0,
        max_steps=5000,
        eval_every=200,
    ),
    dict(
        run_name="runB_more_capacity",
        seed=43,
        context_length=256,
        batch_size=64,
        embedding_dim=384,
        num_blocks=8,
        num_heads=4,
        proj_factor=1.5,
        act_fn="gelu",
        conv1d_kernel_size=4,
        qkv_proj_blocksize=4,
        dropout=0.08,
        lr=1.5e-3,
        weight_decay=0.10,
        grad_clip=1.0,
        max_steps=6000,
        eval_every=200,
    ),
]

for r in runs:
    best_val, ckpt = train_run(text_path=TRUMP_TXT_PATH, use_wandb=True, **r)
    print(f"==> {r['run_name']} best_val={best_val:.4f} ckpt={ckpt}")

########## YOUR SOLUTION HERE ##########

ModuleNotFoundError: No module named 'torch'

## Exercise 4: Utilizing a Pretrained Model

Foundation Models, those pretrained on large amounts of data are more and more important. We can use those models and fine-tune them on our dataset, rather than training them from scratch.
Here are the things to consider:

- Model Selection: Choose a pretrained language model from an online repository. Hint: You can explore platforms like Hugging Face (huggingface.co), which host numerous pretrained models.

- Dataset: Use the Trump dataset with the same training and validation split as in previous exercises. You do not need to use character tokenization.

- Performance Evaluation: Evaluate the performance of the pretrained model on the validation set before and during fine-tuning. Report average-CE-loss as well as an example generated sequence with the same prompt for each epoch.
 
- Fine-tuning: Adjust the learning rate, potentially freeze some layers, train for a few epochs with a framework of your choice (e.g. [lightning](https://lightning.ai/docs/pytorch/stable/), [huggingface](https://huggingface.co/models), ...)

- Computational Resources: Be mindful of the computational demands of pretrained models. You might need access to GPUs. Try to keep the model size at a minimum and go for e.g. distilled versions or other small LMs

- Hyperparameter Tuning: You can experiment with different learning rates and potentially other hyperparameters during fine-tuning but no need to do this in depth

By completing this exercise, you will gain experience with utilizing pretrained models, understanding their capabilities, and the process of fine-tuning. Decreasing the validation loss can be seen as a success for this exercise.

> **Note**: This is a standalone exercise and doesn't build upon the previous tasks.

In [5]:
########## SOLUTION BEGIN ##########

import math
from pathlib import Path
import torch
import inspect
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
    set_seed
)
os.environ["WANDB_DISABLED"] = "true"

TRUMP_TXT_PATH = "trump.txt"
MODEL_NAME = "sshleifer/tiny-gpt2"
OUT_DIR = "./hf_tinygpt2_trump"

SEED = 42
EPOCHS = 3
LR = 5e-4
BLOCK_SIZE = 256
BATCH_SIZE = 8
GRAD_ACC = 4
PROMPT = "Donald Trump said that"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

set_seed(SEED)

# load + split (90/10 like before)
text = Path("trump_train.txt").read_text(encoding="utf-8")
split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]

train_ds = Dataset.from_dict({"text": [train_text]})
val_ds   = Dataset.from_dict({"text": [val_text]})

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"params total={total:,} trainable={trainable:,}")

# tokenize WITHOUT attention_mask to avoid Arrow mismatch
def tok_fn(batch):
    return tok(batch["text"], return_attention_mask=False)

train_tok = train_ds.map(tok_fn, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tok_fn, batched=True, remove_columns=["text"])

def group_texts(ex):
    ids = sum(ex["input_ids"], [])
    ids = ids[: (len(ids)//BLOCK_SIZE)*BLOCK_SIZE]
    blocks = [ids[i:i+BLOCK_SIZE] for i in range(0, len(ids), BLOCK_SIZE)]
    return {"input_ids": blocks, "labels": blocks.copy()}

train_lm = train_tok.map(group_texts, batched=True)
val_lm   = val_tok.map(group_texts, batched=True)

collator = DataCollatorForLanguageModeling(tok, mlm=False)

@torch.no_grad()
def gen(prompt, max_new_tokens=80):
    model.eval()
    inp = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inp,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.9,
        pad_token_id=tok.eos_token_id
    )
    return tok.decode(out[0], skip_special_tokens=True)

ta_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

kwargs = dict(
    output_dir=OUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    save_strategy="no",
    logging_steps=25,
    report_to=["wandb"],   # set [] if you don't want wandb
    fp16=(DEVICE=="cuda"),
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
)

# Use whichever name exists in your installed transformers version
if "evaluation_strategy" in ta_params:
    kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in ta_params:
    kwargs["eval_strategy"] = "epoch"
else:
    print("WARNING: Your transformers version has no evaluation strategy argument; evaluation may need manual calls.")

args = TrainingArguments(**kwargs)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_lm,
    eval_dataset=val_lm,
    tokenizer=tok,
    data_collator=collator,
)

# eval BEFORE
m0 = trainer.evaluate()
print("\n=== BEFORE FINETUNE ===")
print("eval_loss:", m0["eval_loss"], "ppl:", math.exp(m0["eval_loss"]))
print("sample:\n", gen(PROMPT), "\n")

# train + report each epoch
for e in range(1, EPOCHS+1):
    trainer.train()
    m = trainer.evaluate()
    print(f"\n=== AFTER EPOCH {e} ===")
    print("eval_loss:", m["eval_loss"], "ppl:", math.exp(m["eval_loss"]))
    print("sample:\n", gen(PROMPT), "\n")

########## YOUR SOLUTION HERE ##########

DEVICE: cuda
params total=102,714 trainable=102,714


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 70.06 examples/s]
/tmp/ipykernel_92139/3249937457.py:110: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


RuntimeError: You specified `report_to='wandb'` but also set the `WANDB_DISABLED` environment variable.
This disables wandb logging, even though it was explicitly requested.

- To enable wandb logging: unset `WANDB_DISABLED`.
- To disable logging: use `report_to='none'`.

Note: WANDB_DISABLED is deprecated and will be removed in v5.

## Exercise 5: The Memory-Matrix Capacity - Associative Recall

One of the central claims of the xLSTM paper is that the **mLSTM** (matrix LSTM) possesses a superior memory capacity compared to the **sLSTM** (scalar LSTM) due to its matrix cell state $C_t$. While sLSTM manages a scalar memory similar to traditional LSTMs, mLSTM utilizes a matrix memory updated via a covariance rule, effectively functioning as a Key-Value store.

**Task:**
Demonstrate this difference empirically using a synthetic "Associative Recall" task.

We have provided a data generator function `generate_associative_data` below. This function creates sequences of Key-Value pairs followed by a Query Key.
* **Format:** `k1, v1, k2, v2, ..., k_query`
* **Goal:** The model must predict the value associated with `k_query`.
* **Example:** For the sequence `7, 3, 4, 2, ..., 7`, the model should return `3` (recalling that 7 was paired with 3).

**Your Goal:**
1.  **Instantiate models** with comparable parameter counts:
    * **Model A (sLSTM):** A stack consisting only of sLSTM blocks.
    * **Model B (mLSTM):** A stack consisting only of mLSTM blocks.
    * **Model C (LSTM):** (Optional) A standard PyTorch LSTM.
    * **Model D (Transformer):** (Optional) A standard Transformer (e.g., GPT-2 style).
    * *Note:* Keep dimensions small (e.g., `embedding_dim=16`, `num_blocks=2`) for fast iteration.
2.  **Train the models** on the generated data for 1000 steps.
3.  **Vary the head_dim**: try different head dimensions (e.g., 4, 16, 32, 64) while keeping the embedding dimension fixed.
    * *Hypothesis:* How does the matrix memory capacity change as `head_dim` increases?
4.  **Plot your Results**: Visualize the validation accuracy over time for the different configurations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from dacite import from_dict, Config as DaciteConfig
from xlstm import xLSTMBlockStack, xLSTMBlockStackConfig

def generate_associative_data(batch_size, num_pairs, vocab_size, device):
    """
    Generates batch: [k1, v1, k2, v2, ..., k_query]
    Target: v_query
    """
    # Create random pairs
    keys = torch.randint(0, vocab_size, (batch_size, num_pairs))
    vals = torch.randint(0, vocab_size, (batch_size, num_pairs))
    
    # Select a query index for each batch item
    query_indices = torch.randint(0, num_pairs, (batch_size,))
    
    inputs = []
    targets = []
    
    for b in range(batch_size):
        # Interleave keys and values
        seq = torch.stack((keys[b], vals[b]), dim=1).flatten() 
        query_key = keys[b, query_indices[b]]
        target_val = vals[b, query_indices[b]]
        
        # Sequence: k1, v1, k2, v2, ..., query_key
        inputs.append(torch.cat([seq, query_key.unsqueeze(0)]))
        targets.append(target_val)
        
    return torch.stack(inputs).to(device), torch.stack(targets).to(device)

# 1. Configuration - adjust to your liking
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB_SIZE = 64
NUM_PAIRS = 16 
EMBED_DIM = 16 
BATCH_SIZE = 512
STEPS = 1000

########## SOLUTION BEGIN ##########


########## YOUR SOLUTION HERE ##########

## Exercise 6: Visualizing Learned Positional Encodings

In Exercise 5, we saw that the **Transformer** required explicit `pos_embedding` parameters to function, whereas the **xLSTM** models did not. This is because Transformers process data in parallel (permutation invariant), while xLSTMs process data sequentially (time is implicit).

Since we used **learnable** positional embeddings for the Transformer, the model had to *discover* how to represent "position" from scratch during training.

**Task:**
1.  Extract the learned positional embedding weights from your trained model (from Exercise 5, the pretrained one from Exercise 4, or the one from Assignment 4).
2.  Compute the **Cosine Similarity Matrix** between all positions.
    * *Goal:* We want a matrix $M$ of size $(T \times T)$ where $M_{i,j}$ represents how similar the embedding at position $i$ is to position $j$.
3.  Visualize this matrix - What do we observe?

In [ ]:
########## SOLUTION BEGIN ##########


########## YOUR SOLUTION HERE ##########